In [1]:
# ==========================================================
# Breast Cancer Prediction Using Machine Learning
# Algorithms: Logistic Regression, Random Forest, SVM
# Dataset: Wisconsin Diagnostic Breast Cancer (WDBC)
# ==========================================================

# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve
)

In [2]:
# ----------------------------------------------------------
# 1. Load Dataset
# ----------------------------------------------------------

data = load_breast_cancer()

df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target

print("="*50)
print("Dataset Shape :", df.shape)
print(df.head())



Dataset Shape : (569, 31)
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst texture  worst perimeter  wo

In [3]:
# ----------------------------------------------------------
# 2. Exploratory Data Analysis
# ----------------------------------------------------------

plt.figure(figsize=(5,4))
sns.countplot(x=df["target"])
plt.title("Benign vs Malignant")
plt.xticks([0,1],["Malignant","Benign"])
plt.savefig("class_distribution.png")
plt.close()

# Correlation Heatmap
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.savefig("correlation_heatmap.png")
plt.close()



In [4]:
# ----------------------------------------------------------
# 3. Data Preprocessing
# ----------------------------------------------------------

X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



In [5]:
# ----------------------------------------------------------
# 4. Create Models
# ----------------------------------------------------------

lr = LogisticRegression(max_iter=500)

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

svm = SVC(
    kernel="rbf",
    probability=True,
    random_state=42
)



In [6]:
# ----------------------------------------------------------
# 5. Train Models
# ----------------------------------------------------------

lr.fit(X_train_scaled, y_train)
rf.fit(X_train, y_train)
svm.fit(X_train_scaled, y_train)



SVC(probability=True, random_state=42)

In [7]:
# ----------------------------------------------------------
# 6. Predictions
# ----------------------------------------------------------

lr_pred = lr.predict(X_test_scaled)
rf_pred = rf.predict(X_test)
svm_pred = svm.predict(X_test_scaled)



In [8]:
# ----------------------------------------------------------
# 7. Evaluation Function
# ----------------------------------------------------------

def evaluate(name, y_true, pred, prob):

    acc = accuracy_score(y_true, pred)
    pre = precision_score(y_true, pred)
    rec = recall_score(y_true, pred)
    f1 = f1_score(y_true, pred)
    auc = roc_auc_score(y_true, prob)

    print(f"\n{name}")
    print("-"*40)
    print("Accuracy :", round(acc,4))
    print("Precision:", round(pre,4))
    print("Recall   :", round(rec,4))
    print("F1 Score :", round(f1,4))
    print("ROC AUC  :", round(auc,4))

    return [name, acc, pre, rec, f1, auc]


results = []

results.append(
    evaluate(
        "Logistic Regression",
        y_test,
        lr_pred,
        lr.predict_proba(X_test_scaled)[:,1]
    )
)

results.append(
    evaluate(
        "Random Forest",
        y_test,
        rf_pred,
        rf.predict_proba(X_test)[:,1]
    )
)

results.append(
    evaluate(
        "Support Vector Machine",
        y_test,
        svm_pred,
        svm.predict_proba(X_test_scaled)[:,1]
    )
)




Logistic Regression
----------------------------------------
Accuracy : 0.9825
Precision: 0.9861
Recall   : 0.9861
F1 Score : 0.9861
ROC AUC  : 0.9954

Random Forest
----------------------------------------
Accuracy : 0.9561
Precision: 0.9589
Recall   : 0.9722
F1 Score : 0.9655
ROC AUC  : 0.9931

Support Vector Machine
----------------------------------------
Accuracy : 0.9825
Precision: 0.9861
Recall   : 0.9861
F1 Score : 0.9861
ROC AUC  : 0.995


In [9]:
# ----------------------------------------------------------
# 8. Results Table
# ----------------------------------------------------------

result_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC"
    ]
)

print("\nFinal Comparison")
print(result_df)

result_df.to_csv("model_results.csv", index=False)




Final Comparison
                    Model  Accuracy  Precision    Recall        F1   ROC_AUC
0     Logistic Regression  0.982456   0.986111  0.986111  0.986111  0.995370
1           Random Forest  0.956140   0.958904  0.972222  0.965517  0.993056
2  Support Vector Machine  0.982456   0.986111  0.986111  0.986111  0.995040


In [10]:
# ----------------------------------------------------------
# 9. Accuracy Comparison Graph
# ----------------------------------------------------------

plt.figure(figsize=(7,5))
sns.barplot(
    data=result_df,
    x="Model",
    y="Accuracy"
)

plt.ylim(0.90,1.0)
plt.title("Accuracy Comparison")
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig("accuracy_comparison.png")
plt.close()



In [11]:
# ----------------------------------------------------------
# 10. ROC Curve
# ----------------------------------------------------------

plt.figure(figsize=(6,6))

models = [
    ("LR", lr.predict_proba(X_test_scaled)[:,1]),
    ("RF", rf.predict_proba(X_test)[:,1]),
    ("SVM", svm.predict_proba(X_test_scaled)[:,1])
]

for name, prob in models:
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, label=name)

plt.plot([0,1],[0,1],"k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.savefig("roc_curve.png")
plt.close()



In [12]:
# ----------------------------------------------------------
# 11. Confusion Matrix (SVM)
# ----------------------------------------------------------

cm = confusion_matrix(y_test, svm_pred)

plt.figure(figsize=(5,4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("SVM Confusion Matrix")
plt.savefig("confusion_matrix.png")
plt.close()



In [13]:
# ----------------------------------------------------------
# 12. Feature Importance (Random Forest)
# ----------------------------------------------------------

importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
)

top10 = importance.sort_values(
    ascending=False
).head(10)

plt.figure(figsize=(8,5))
top10.sort_values().plot(kind="barh")
plt.title("Top 10 Important Features")
plt.tight_layout()
plt.savefig("feature_importance.png")
plt.close()



In [14]:
# ----------------------------------------------------------
# 13. Cross Validation
# ----------------------------------------------------------

cv = cross_val_score(
    svm,
    X_train_scaled,
    y_train,
    cv=10
)

print("\nSVM Cross Validation Mean :", round(cv.mean(),4))
print("SVM Standard Deviation    :", round(cv.std(),4))




SVM Cross Validation Mean : 0.9738
SVM Standard Deviation    : 0.0272


In [15]:
# ----------------------------------------------------------
# 14. Best Model
# ----------------------------------------------------------

best = result_df.sort_values(
    by="Accuracy",
    ascending=False
).iloc[0]

print("\n"+"="*50)
print("BEST MODEL :", best["Model"])
print("Accuracy   :", round(best["Accuracy"],4))
print("="*50)

print("\nAll graphs and CSV file have been saved successfully.")


BEST MODEL : Logistic Regression
Accuracy   : 0.9825

All graphs and CSV file have been saved successfully.
